# SpatialArtifacts: VisiumHD Tutorial

This notebook demonstrates the `SpatialArtifacts` workflow for detecting and classifying edge artifacts in 10x Genomics VisiumHD spatial transcriptomics data.

The workflow has two steps:
1. **`detect_edge_artifacts()`** — identifies outlier bins using MAD-based detection combined with morphological image processing and buffer zone geometry
2. **`classify_edge_artifacts()`** — assigns hierarchical labels based on location (edge vs interior) and size (large vs small)

**Key difference from Standard Visium:** VisiumHD uses a square grid at much higher resolution (16µm or 8µm bins vs 55µm spots). Parameters are specified in physical units (µm) rather than bin counts, and `min_spots` in the classification step must be scaled accordingly.

**Data:** Replace `UNZIP_DIR` and `SAMPLE_ID` with your own VisiumHD data paths.

**Reference:** He et al. (2026), SpatialArtifacts package.

## Installation

```bash
pip install spatial-artifacts
```

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
from pathlib import Path

from spatial_artifacts import detect_edge_artifacts, classify_edge_artifacts

## Parameters

Set your data path, resolution, and detection parameters here.

**Parameter scaling for VisiumHD:** The `min_spots` threshold in `classify_edge_artifacts()` must be scaled relative to Standard Visium (55µm) to represent equivalent physical artifact sizes:

```
min_spots_HD = min_spots_visium x (55 / bin_size_um)^2

# For min_spots_visium = 20:
# VisiumHD 16um:  20 x (55/16)^2 ~ 236 bins
# VisiumHD  8um:  20 x (55/8)^2  ~ 945 bins
```

In [ ]:
UNZIP_DIR = Path("/path/to/your/visiumhd/data/")
RESOLUTION = "16um"          # "16um" or "8um"
SAMPLE_ID = "your_sample_id"
MAD_THRESHOLD = 3
BUFFER_WIDTH_UM = 160
MIN_CLUSTER_AREA_UM2 = 1280
MIN_SPOTS = 200              # Scale this based on resolution (see note above)
NAME = "edge_artifact"

## Load Data

VisiumHD data is stored in `.h5` format with spatial coordinates in a `.parquet` file.

In [ ]:
data_dir = UNZIP_DIR / "binned_outputs" / f"square_{RESOLUTION}"
h5_file = data_dir / "filtered_feature_bc_matrix.h5"
parquet_file = data_dir / "spatial" / "tissue_positions.parquet"

print(f"Loading VisiumHD sample: {SAMPLE_ID}")
adata = sc.read_10x_h5(h5_file)
adata.var_names_make_unique()
print(f"Loaded: {adata.n_obs} bins, {adata.n_vars} genes")

## Spatial Coordinates

VisiumHD spatial coordinates are stored in a Parquet file (unlike Standard Visium which uses a CSV).

In [ ]:
print("Loading spatial coordinates...")
pos_df = pq.read_table(parquet_file).to_pandas().set_index("barcode")

common = adata.obs_names.intersection(pos_df.index)
adata = adata[common].copy()
pos_df = pos_df.loc[adata.obs_names]

adata.obs["in_tissue"] = pos_df["in_tissue"].astype(bool).values
adata.obs["array_row"] = pos_df["array_row"].values
adata.obs["array_col"] = pos_df["array_col"].values
adata.obs["pxl_row_in_fullres"] = pos_df["pxl_row_in_fullres"].values
adata.obs["pxl_col_in_fullres"] = pos_df["pxl_col_in_fullres"].values

adata = adata[adata.obs["in_tissue"]].copy()
print(f"In-tissue bins: {adata.n_obs}")

## QC Metrics

In [ ]:
sc.pp.calculate_qc_metrics(adata, inplace=True)
adata.obs["sum_umi"] = adata.obs["total_counts"]
adata.obs["sample_id"] = SAMPLE_ID

print(f"array_row range: {adata.obs['array_row'].min()} - {adata.obs['array_row'].max()}")
print(f"array_col range: {adata.obs['array_col'].min()} - {adata.obs['array_col'].max()}")

## Step 1: Detect Edge Artifacts

For VisiumHD, `detect_edge_artifacts()` uses a two-phase approach:
1. **Buffer zone detection**: identifies bins within `buffer_width_um` micrometers of the tissue boundary
2. **Morphological cleaning**: same fill/outline/star operations as Standard Visium, scaled to VisiumHD resolution

Key parameters for VisiumHD:
- `resolution`: bin size, either `"16um"` or `"8um"` (required)
- `buffer_width_um`: edge buffer zone width in micrometers
- `min_cluster_area_um2`: minimum cluster area in µm² (converted to bins internally)

In [ ]:
adata = detect_edge_artifacts(
    adata,
    platform="visiumhd",
    resolution=RESOLUTION,
    qc_metric="sum_umi",
    samples="sample_id",
    mad_threshold=MAD_THRESHOLD,
    buffer_width_um=BUFFER_WIDTH_UM,
    min_cluster_area_um2=MIN_CLUSTER_AREA_UM2,
    col_x="array_col",
    col_y="array_row",
    name=NAME,
    verbose=True,
)

print("\nRaw edge detection:")
print(adata.obs[f"{NAME}_edge"].value_counts())

## Step 2: Classify Edge Artifacts

`classify_edge_artifacts()` applies the same 2x2 hierarchical logic as Standard Visium:
- **Location**: edge vs interior  
- **Size**: large (`> min_spots`) vs small

**Important:** `min_spots` here is in bins, not physical spots. Use the scaling formula in the Parameters cell above.

In [ ]:
adata = classify_edge_artifacts(
    adata,
    min_spots=MIN_SPOTS,
    name=NAME,
    verbose=True,
)

# Filter zero-count bins
adata = adata[adata.obs["total_counts"] > 0].copy()
print(f"After filtering zero-count bins: {adata.n_obs}")

print("\nClassification summary:")
print(adata.obs[f"{NAME}_classification"].value_counts())

## Visualization

Two panels showing:
- **A**: Total UMI counts (log10 scale)
- **B**: SpatialArtifacts hierarchical classification

Note: VisiumHD has ~480k bins at 16µm resolution, so point size is smaller (`s=0.1`).

In [ ]:
artifact_colors = {
    "not_artifact":            "lightgray",
    "small_edge_artifact":     "#FFB347",
    "large_edge_artifact":     "#FF4500",
    "small_interior_artifact": "#00CED1",
    "large_interior_artifact": "#0047AB",
}

obs = adata.obs.copy()
x = obs["pxl_col_in_fullres"].values
y = -obs["pxl_row_in_fullres"].values

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Total UMI (log10)
log_umi = np.log10(obs["sum_umi"].values + 1)
sc_a = axes[0].scatter(x, y, c=log_umi, cmap="magma", s=0.1, alpha=0.8)
plt.colorbar(sc_a, ax=axes[0], label="log10(UMI)")
axes[0].set_title("A. Total UMI Distribution", fontweight="bold")
axes[0].axis("off")
axes[0].set_aspect("equal")

# Panel B: SpatialArtifacts classification
cls = obs[f"{NAME}_classification"].values
order = np.argsort(cls == "not_artifact")[::-1]
colors_b = [artifact_colors.get(c, "lightgray") for c in cls[order]]
axes[1].scatter(x[order], y[order], c=colors_b, s=0.1, alpha=0.8)
axes[1].set_title("B. SpatialArtifacts Classification", fontweight="bold")
axes[1].axis("off")
axes[1].set_aspect("equal")

for label, color in artifact_colors.items():
    if label in cls:
        axes[1].scatter([], [], c=color, label=label.replace("_", " "), s=20)
axes[1].legend(loc="lower right", fontsize=7, framealpha=0.8)

plt.suptitle(f"Sample: {SAMPLE_ID} ({RESOLUTION})", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## Filtering (Optional)

Remove edge artifact bins for downstream analysis.

In [ ]:
bins_to_keep = ~adata.obs[f"{NAME}_classification"].isin(
    ["large_edge_artifact", "small_edge_artifact"]
)
adata_filtered = adata[bins_to_keep].copy()

print(f"Original bins:   {adata.n_obs}")
print(f"After filtering: {adata_filtered.n_obs}")
print(f"Removed:         {adata.n_obs - adata_filtered.n_obs} bins")

## Session Info

In [ ]:
sc.logging.print_versions()